## Ch14. Neural networks Forecasting: Principles & Practice (Python Edition) Extracted from: fpppy-14-neural-networks.qmd

## [Setup] Imports & Configuration --- Setup / Hidden in slides ---

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from neuralforecast import NeuralForecast
from neuralforecast.models import (
    MLP, NHITS, RNN, NBEATS, BiTCN, AutoNHITS
)
from neuralforecast.losses.pytorch import MAE, MSE, DistributionLoss
from neuralforecast.utils import AirPassengersPanel
from utilsforecast.plotting import plot_series
plt.rcParams.update({"figure.figsize": (7, 3.5)})

2026-09-20 06:20:28,960	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


2026-09-20 06:20:29,347	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


ImportError: cannot import name 'AutoNHITS' from 'neuralforecast.models' (/Data2/data/jc25/forecast/fpppy-labs/.venv/lib/python3.11/site-packages/neuralforecast/models/__init__.py)

## [Slide 5] 14.2 MLP for Air Passengers

In [2]:
test_mask = AirPassengersPanel["ds"] >= "1960"
Y_train_df = AirPassengersPanel[~test_mask]
Y_test_df  = AirPassengersPanel[test_mask].reset_index(drop=True)

model = MLP(
    h=12,            # forecast horizon
    input_size=24,   # 24 months of history
    scaler_type="robust",
)
nf = NeuralForecast(models=[model], freq="M")
nf.fit(df=Y_train_df)
forecasts = nf.predict()

NameError: name 'AirPassengersPanel' is not defined

## [Slide 8] 14.3 NHITS and RNN Example

In [3]:
models = [
    NHITS(h=12, input_size=24, scaler_type="robust"),
    RNN(h=12,   input_size=24, scaler_type="robust"),
]
nf = NeuralForecast(models=models, freq="M")
nf.fit(df=Y_train_df)
forecasts = nf.predict()

fig, axes = plt.subplots(2, sharex=True, figsize=(8, 5.5))
plot_series(
    AirPassengersPanel, forecasts,
    xlabel="", ylabel="",
    palette="black_and_2color", rm_legend=False,
    ax=axes, legend_loc="outside lower center",
)
axes[-1].set(xlabel="Month [1M]")
fig.suptitle("Number of passengers for different airlines",
             x=.54, fontsize=12)
fig.supylabel("Passengers")

Seed set to 1


Seed set to 1


NameError: name 'Y_train_df' is not defined

## [Slide 10] 14.4 Scaling the Data Poor performance without scaling:

In [4]:
model = MLP(h=12, input_size=24, scaler_type="identity")

Seed set to 1


## [Slide 12] 14.5 Optimisation Objectives

In [5]:
model = NHITS(h=12, input_size=24,
              loss=MSE(), scaler_type="robust")

NameError: name 'MSE' is not defined

## [Slide 14] 14.5 Probabilistic Optimisation Objectives

In [6]:
model = NHITS(
    h=12,
    input_size=24,
    loss=DistributionLoss(distribution="Normal"),
    scaler_type="robust",
)
nf = NeuralForecast(models=[model], freq="M")
nf.fit(df=Y_train_df)

NameError: name 'DistributionLoss' is not defined

Produces prediction intervals (80%, 90% by default)

In [7]:
forecasts = nf.predict()

Exception: You must pass a DataFrame or have one stored.

## [Slide 16] 14.6 Exogenous Variables

In [8]:
df        = pd.read_csv("data/EPF_FR_BE.csv", parse_dates=["ds"])
static_df = pd.read_csv("data/EPF_FR_BE_static.csv")
futr_df   = pd.read_csv("data/EPF_FR_BE_futr.csv",
                         parse_dates=["ds"])

FileNotFoundError: [Errno 2] No such file or directory: 'data/EPF_FR_BE.csv'

## [Slide 18] 14.6 BiTCN with Exogenous Variables

In [9]:
horizon = 24   # day-ahead hourly forecast

model = BiTCN(
    h=horizon,
    input_size=5 * horizon,
    futr_exog_list=["gen_forecast", "week_day"],
    hist_exog_list=["system_load"],
    stat_exog_list=["market_0", "market_1"],
    scaler_type="robust",
    max_steps=300,
)
nf = NeuralForecast(models=[model], freq="H")
nf.fit(df=df, static_df=static_df)
forecasts = nf.predict(futr_df=futr_df)

Seed set to 1


NameError: name 'df' is not defined

## [Slide 20] 14.7 Hyperparameter Optimisation

In [10]:
from ray import tune

nhits_config = {
    **AutoNHITS.get_default_config(h=12, backend="ray"),
    "random_seed": tune.randint(1, 10),
    "n_pool_kernel_size": tune.choice(
        [[2, 2, 2], [16, 8, 1]]
    ),
    "max_steps": tune.choice([100]),
}
model = AutoNHITS(h=12, num_samples=10,
                  config=nhits_config)
nf = NeuralForecast(models=[model], freq="M")
nf.fit(df=Y_train_df)
forecasts = nf.predict()

NameError: name 'AutoNHITS' is not defined

## [Slide 22] 14.7 Cross-Validation for Neural Networks

In [11]:
nf = NeuralForecast(
    models=[NHITS(h=12, input_size=24,
                  loss=MAE(), max_steps=500)],
    freq="MS"
)

NameError: name 'MAE' is not defined

Time-series cross-validation with n_windows expanding windows

In [12]:
cv_df = nf.cross_validation(
    df=Y_train_df,
    n_windows=3,
    step_size=12,
)

NameError: name 'Y_train_df' is not defined